# Proyecto — Casos de estudio de raíces de ecuaciones + Monte Carlo (U2_T4)

**Mecatrónica · Métodos Numéricos**

Cada **equipo** elige **un caso de estudio de ingeniería** (una ecuación o un sistema de
raíces) y lo analiza con una **simulación de Monte Carlo** para decidir, con evidencia,
**qué método numérico conviene** en un sistema real con incertidumbre.

Al terminar tendrás:
1. Tu **librería de métodos** implementada por el equipo.
2. Los **gráficos** de la simulación (incluido el **box plot** comparativo).
3. Una **calificación automática** de la parte de métodos y 5 preguntas de criterio.

---

> ### 👉 Antes de empezar
> - **Solo el jefe de equipo** ejecuta las celdas de **selección de tema** e **integrantes**.
> - El jefe **descarga el repositorio** y **comparte la carpeta de Drive** con su equipo.
> - El jefe **inicia sesión con su cuenta institucional** (`@campus.tecnm.mx`).


### INICIA SESIÓN CON TU CORREO INSTITUCIONAL


In [ ]:
#@title ⚙️ Configuración general — ejecuta esta celda **PRIMERO** (no modificar)
# ============================================================
# Detecta si corre en Google Colab o en tu computadora, monta Drive y
# carga el motor de la tarea (ver MODO_MOTOR más abajo).
# ============================================================
import base64, importlib, os, subprocess, sys, zlib, json

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

try:
    import google.colab
    ES_COLAB = True
except ImportError:
    ES_COLAB = False

# --- Repositorio de la tarea  <-- EDITAR SOLO SI CAMBIA EL REPO ------
REPO_URL = 'https://github.com/ITH-MGL-MN/U2_T4_Casos_de_estudio.git'
REPO_NOMBRE = 'U2_T4_Casos_de_estudio'

# --- Solo en tu computadora: de dónde sale el motor ---
#   "modulos" -> profe/** tal cual, importado como paquete (para depurar)
#   "bundle"  -> grader_local.py, el motor APLANADO (forma exacta de Colab)
#   "blob"    -> grader_ofuscado.txt, lo que recibe el alumno
MODO_MOTOR = "modulos"

if ES_COLAB:
    BASE = "/content/drive/MyDrive/CursoMN"
    REPO = os.path.join(BASE, REPO_NOMBRE)
else:
    REPO = os.getcwd()
    for _ in range(4):
        if os.path.exists(os.path.join(REPO, "grader_local.py")):
            break
        _padre = os.path.dirname(REPO)
        if _padre == REPO:
            break
        REPO = _padre
    if not os.path.exists(os.path.join(REPO, "grader_local.py")):
        REPO = r"o:\Mi unidad\CursoMN\U2_T4_Casos_de_estudio"

MODO = "blob" if ES_COLAB else MODO_MOTOR
print("Modo          :", "Google Colab" if ES_COLAB else "PC local")
print("Carpeta tarea :", REPO)

# ============================================================
# Montar Drive y traer el repositorio  (solo en Colab)
# ============================================================
if ES_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(BASE, exist_ok=True)
    try:
        if not os.path.exists(REPO):
            subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
        else:
            subprocess.run(["git", "-C", REPO, "pull"], check=True)
        print("✅ Repositorio listo en Drive.")
    except Exception as _e:
        print("⚠️ No se pudo clonar/actualizar el repositorio:", _e)
        print("   Si la carpeta ya existe en tu Drive puedes continuar.")
else:
    print("Modo local: no se monta Drive.")

# ============================================================
# Librerías necesarias
# ============================================================
for _pkg in ("numpy", "scipy", "sympy", "matplotlib", "requests", "ipywidgets", "plotly"):
    try:
        importlib.import_module(_pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])

# ============================================================
# Motor de la tarea
# ============================================================
# El motor vive aparte del cuaderno y este recibe solo la API que el motor
# declara en su `__all__`. En Colab se usa el blob; en local, MODO_MOTOR.
sys.path.insert(0, REPO)

if MODO == "modulos":
    from profe.ui.cuaderno import *      # noqa: F401,F403
else:
    if MODO == "blob":
        _ruta = os.path.join(REPO, "grader_ofuscado.txt")
        with open(_ruta, encoding="utf-8") as f:
            _blob = f.read().strip()
        codigo = zlib.decompress(base64.b64decode(_blob)).decode("utf-8")
    else:
        _ruta = os.path.join(REPO, "grader_local.py")
        with open(_ruta, encoding="utf-8") as f:
            codigo = f.read()
    _MOTOR = {}
    try:
        exec(codigo, _MOTOR)
    except Exception as _e:
        raise RuntimeError(
            "No pude cargar el motor desde %s.\n"
            "En local: ejecuta  python tools/build.py  para generarlo.\n"
            "Detalle: %r" % (_ruta, _e))
    globals().update({_n: _MOTOR[_n] for _n in _MOTOR["__all__"]})

print("Motor cargado :", MODO)

# ============================================================
# Preludio estilo MATLAB + robomat
# ============================================================
sys.path.insert(0, os.path.join(REPO, "lib"))
with open(os.path.join(REPO, "lib", "matlab_like.py"), encoding="utf-8") as f:
    exec(f.read(), globals())
from robomat import M, I, sin, cos, tan, var, rad, deg, evaluar, mostrar, jacobiano
print("Preludio MATLAB:", os.path.join(REPO, "lib", "matlab_like.py"))

# ============================================================
# Herramientas de visualización (gráficas interactivas plotly)
# ============================================================
# Viven en tools/visualizar.py, FUERA del motor: no forman parte del sistema
# de calificación, así que puedes leerlas y reutilizarlas.
sys.path.append(os.path.join(REPO, "tools"))
try:
    import visualizar
    importlib.reload(visualizar)
except ImportError as _e:
    raise RuntimeError(
        "No encontré tools/visualizar.py, que las gráficas interactivas usan.\n"
        "En la carpeta de la tarea ese archivo debe estar junto al cuaderno.\n"
        "Detalle: %r" % (_e,))
print("Visualización :", os.path.join(REPO, "tools", "visualizar.py"))

# ============================================================
# Estado del equipo (se guarda en Drive para no perderlo)
# ============================================================
RUTA_ESTADO = os.path.join(REPO, "equipo_u2t4.json")
EQUIPO = None
if os.path.exists(RUTA_ESTADO):
    try:
        with open(RUTA_ESTADO, encoding="utf-8") as f:
            EQUIPO = json.load(f)
        print("📄 Estado del equipo recuperado:", EQUIPO.get("tema"), "-", EQUIPO.get("estado"))
    except Exception as e:
        print("No se pudo leer el estado previo:", e)

import requests

def _post(accion, **campos):
    """Llama a Apps Script y devuelve la carga útil o lanza el error del servidor."""
    payload = {"accion": accion, "token": WEBHOOK_TOKEN}
    payload.update(campos)
    r = requests.post(APPS_SCRIPT_URL, json=payload, timeout=40)
    data = r.json()
    if data.get("status") != "ok":
        raise RuntimeError(data.get("message", "Error desconocido del servidor"))
    return data.get("data", {})

def _guardar_estado():
    with open(RUTA_ESTADO, "w", encoding="utf-8") as f:
        json.dump(EQUIPO, f, ensure_ascii=False, indent=1)

print("✅ Entorno listo. Ahora ejecuta la celda de identificación.")


## Identifícate (jefe de equipo)

Te identificarás con la cuenta institucional con la que montaste Google Drive.
Después escribe tu **número de control** y se validará contra la lista del grupo.

In [ ]:
# @title Identificación automática del jefe de equipo
import ipywidgets as widgets
from IPython.display import display, HTML

if ES_COLAB:
    from google.colab import auth
    import google.auth, google.auth.transport.requests
    try:
        auth.authenticate_user()
        creds, _ = google.auth.default()
        creds.refresh(google.auth.transport.requests.Request())
        info = requests.get(
            'https://www.googleapis.com/drive/v3/about?fields=user',
            headers={'Authorization': 'Bearer ' + creds.token}, timeout=20).json()
        CORREO = info.get('user', {}).get('emailAddress', '')
    except Exception as e:
        print('⚠️ No se pudo obtener tu correo automáticamente.', e)
        CORREO = ''
else:
    CORREO = 'm16330887@hermosillo.tecnm.mx'
    # CORREO = input('Correo institucional (pruebas): ').strip()

if CORREO and '@' in CORREO and CORREO.lower().endswith('.tecnm.mx'):
    print('✅ Identificado: ' + CORREO)
elif CORREO:
    display(HTML('<p style="color:#b00020;background:#ffe0e0;padding:12px;border-radius:8px;'
                 'font-weight:bold">⛔ Este correo no es institucional.<br>'
                 'Usa tu cuenta @campus.tecnm.mx.</p>'))
else:
    print('No se pudo identificar el correo. Vuelve a intentar.')

# Número de control sugerido a partir del correo (si trae los 8 dígitos)
_sug = extraer_nc(CORREO) or (EQUIPO.get('jefe') if EQUIPO else '')
NC_JEFE = input('Número de control del jefe de equipo [%s]: ' % _sug).strip() or _sug
print('NC del jefe:', NC_JEFE)

# Validación contra la lista del grupo
if NC_JEFE:
    _al = _post('proyecto_equipo', NC=NC_JEFE, correo=CORREO)
    if _al.get('equipo'):
        EQUIPO = _al['equipo']
        _guardar_estado()
        print('✅ Este equipo YA tiene el tema %s (%s).' % (EQUIPO['tema'], EQUIPO['estado']))
    else:
        print('✅ NC válido y sin equipo asignado. Sigue con la selección de tema.')
else:
    print('⛔ Escribe tu número de control.')

✅ Identificado: jesus.medinag@hermosillo.tecnm.mx
Número de control del jefe de equipo [16330887]: 
NC del jefe: 16330887


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

# Guarda el Notebook
Ve a `Archivo → Guardar una copia en Drive` y guárdalo en
`CursoMN/U2_T4_Casos_de_estudio` para conservar tu trabajo.

---
# 1. Elige tu caso de estudio

Esta tarea **no** usa semilla por número de control: **el equipo elige su tema**.

| Paso | Quién | Celda |
|---|---|---|
| 1 | Jefe | **Ver temas disponibles** |
| 2 | Jefe | **Reservar tema** ← *queda bloqueado, ya no se puede cambiar* |
| 3 | Jefe | **Integrantes** |
| 4 | Jefe | **Registrar equipo** |
| 5 | Todo el equipo | Trabajar en el notebook |

> ⚠️ **Una vez reservado el tema, ya no se puede cambiar.** Si te equivocaste, usa la
> celda **Eliminar equipo** (hasta abajo) y vuelve a empezar.

**Bloques según el tamaño del equipo**

| Tamaño | Bloque | Temas |
|---|---|---|
| 2 | A — Básico | T1 – T4 |
| 3 | B — Intermedio | T5 – T9 |
| 4 | C — Estándar mecatrónica | T10 – T15 |
| 5 | D — Avanzado | T16 – T17 |
| 6 | E — Integrador | T18 – T19 |

El **catálogo completo** (con ecuaciones, incertidumbres y desafíos) está en el
`README.md` de la carpeta del proyecto.

In [ ]:
# @title Ver temas disponibles
from IPython.display import display, HTML

_datos = _post('proyecto_temas')
TEMAS = _datos['temas']
LIBRES = [t for t in TEMAS if t['disponible']]

print('Temas LIBRES: %d de %d\n' % (len(LIBRES), len(TEMAS)))
for t in TEMAS:
    if t['disponible']:
        print('  %-4s (%d integrantes) %s' % (t['id'], t['tamano'], t['titulo']))
print()
for t in TEMAS:
    if not t['disponible']:
        print('  %-4s OCUPADO (%s) por NC %s' % (t['id'], t['estado'], t['jefe']))

if EQUIPO:
    print('\n⚠️ Tu equipo ya tiene el tema %s. No necesitas reservar.' % EQUIPO['tema'])

In [ ]:
# @title Reservar tema (BLOQUEA el tema para tu equipo)
SL_TEMA = widgets.Dropdown(
    options=[('%s — %s' % (t['id'], t['titulo']), t['id']) for t in LIBRES],
    description='Tu tema:', layout=widgets.Layout(width='620px'))
BT_RESERVAR = widgets.Button(description='Verificar disponibilidad y reservar',
                             button_style='warning')
OUT_TEMA = widgets.Output()

def _reservar(_):
    global EQUIPO
    with OUT_TEMA:
        OUT_TEMA.clear_output()
        if EQUIPO:
            print('⚠️ Tu equipo ya tiene el tema %s (%s).' % (EQUIPO['tema'], EQUIPO['estado']))
            return
        if not NC_JEFE:
            print('⛔ Primero ejecuta la celda de identificación.')
            return
        try:
            res = _post('proyecto_reservar', NC_Jefe=NC_JEFE, correo=CORREO, tema=SL_TEMA.value)
        except Exception as e:
            print('⛔ No se pudo reservar:', e)
            print('   Vuelve a ejecutar la celda anterior para ver los temas libres.')
            return
        EQUIPO = res['equipo']
        _guardar_estado()
        print('✅', res['mensaje'])
        print('   Integrantes requeridos:', EQUIPO['tamano'])
        print('   Sigue con la celda de INTEGRANTES.')

BT_RESERVAR.on_click(_reservar)
display(widgets.VBox([SL_TEMA, BT_RESERVAR, OUT_TEMA]))

---
# 2. Integrantes del equipo

Aparecerán **tantos selectores como integrantes** tenga tu tema. El **jefe ya viene
seleccionado** (es quien inició sesión). Elige a tus compañeros y luego registra.

> 🔹 En los temas **T2** y **T4** también se permite registrar con **1 solo integrante**
> (el jefe). En ese caso el selector *Nº integrantes* se pone en **1**.


In [ ]:
# @title Integrantes (un selector por integrante)
SEL = {}

if not EQUIPO:
    print('⛔ Primero reserva tu tema (celda anterior).')
else:
    _al = _post('proyecto_alumnos')['alumnos']
    _opciones = [(a['nombre'], a['nc']) for a in _al]
    _opciones.sort()

    N_MAX = int(EQUIPO.get('tamano', 2) or 2)          # máximo del tema
    N_MIN = int(EQUIPO.get('minimo', N_MAX) or N_MAX)  # mínimo permitido (1 si el tema lo admite)
    N_MIN = max(1, min(N_MIN, N_MAX))

    print('Tema %s — %s' % (EQUIPO['tema'], EQUIPO['titulo']))
    if N_MIN < N_MAX:
        print('Este tema se puede registrar con 1 integrante (solo el jefe) o hasta %d.' % N_MAX)
    else:
        print('Este tema requiere %d integrante(s).' % N_MAX)

    W_NINT = widgets.Dropdown(options=list(range(N_MIN, N_MAX + 1)), value=N_MAX,
                              description='Nº integrantes:',
                              layout=widgets.Layout(width='340px'))
    OUT_INT = widgets.Output()

    def _reconstruir_integrantes(_=None):
        """Dibuja el selector del jefe + (n-1) compañeros según W_NINT."""
        global SEL
        _ya = [i['nc'] for i in EQUIPO.get('integrantes', [])]
        n = int(W_NINT.value)
        _nuevo = {}
        for i in range(n):
            if i == 0:
                _nuevo[i] = widgets.Dropdown(options=_opciones, value=NC_JEFE,
                                             description='Jefe:',
                                             layout=widgets.Layout(width='620px'))
            else:
                _def = _ya[i] if len(_ya) > i else _opciones[0][1]
                _nuevo[i] = widgets.Dropdown(options=_opciones, value=_def,
                                             description='Integrante %d:' % (i + 1),
                                             layout=widgets.Layout(width='620px'))
        SEL = _nuevo
        with OUT_INT:
            OUT_INT.clear_output()
            display(widgets.VBox(list(SEL.values())))

    if N_MIN < N_MAX:
        W_NINT.observe(_reconstruir_integrantes, names='value')
        display(W_NINT)
    _reconstruir_integrantes()
    display(OUT_INT)


In [ ]:
# @title Verificar y registrar equipo
BT_REG = widgets.Button(description='Verificar y registrar equipo', button_style='success')
OUT_REG = widgets.Output()

def _registrar(_):
    global EQUIPO
    with OUT_REG:
        OUT_REG.clear_output()
        try:
            _integrantes = [SEL[i].value for i in range(len(SEL))]
        except Exception:
            print('⛔ Primero ejecuta la celda de integrantes.')
            return
        try:
            res = _post('proyecto_registrar', NC_Jefe=NC_JEFE, correo=CORREO,
                        tema=EQUIPO['tema'], integrantes=_integrantes)
        except Exception as e:
            print('⛔ No se pudo registrar:', e)
            return
        EQUIPO = res['equipo']
        _guardar_estado()
        print('✅', res['mensaje'])
        print('\nEquipo registrado:')
        for i in EQUIPO['integrantes']:
            print('   %s  %s%s' % (i['nc'], i['nombre'], '  (jefe)' if i['esJefe'] else ''))
        print('\n🎉 Listo. Ya no necesitas volver a ejecutar estas celdas.')
        print('   Avanza a la sección 4 con TU caso de estudio.')

BT_REG.on_click(_registrar)
display(widgets.VBox([BT_REG, OUT_REG]))

### Eliminar equipo (solo si hubo un error)

Si se equivocaron de tema o de integrantes, pueden **borrar el equipo** y volver a
empezar. Por seguridad se requieren **dos** condiciones:

1. Cambiar `PERMITIR_ELIMINAR = True` aquí abajo (por defecto `False`).
2. Que **tu** número de control **pertenezca** al equipo.

Así nadie puede borrar el equipo de otros.

In [ ]:
PERMITIR_ELIMINAR = False   # <-- cámbialo a True solo para corregir un error
CONFIRMAR = True            # <-- debe quedar en True para borrar

In [ ]:
# @title Eliminar equipo (requiere PERMITIR_ELIMINAR = True)
if not PERMITIR_ELIMINAR:
    print('🔒 La eliminación está desactivada (PERMITIR_ELIMINAR = False).')
elif not EQUIPO:
    print('No hay equipo registrado en este notebook.')
else:
    try:
        res = _post('proyecto_eliminar', NC_Actual=NC_JEFE, tema=EQUIPO['tema'],
                    confirmar=(CONFIRMAR is True))
        print('🗑️', res['mensaje'])
        EQUIPO = None
        _guardar_estado()
        print('Vuelve a la sección 1 para elegir otro tema.')
    except Exception as e:
        print('⛔ No se pudo eliminar:', e)

---
# 3. Tu caso de estudio

Aquí abajo aparece **solo tu proyecto** (el catálogo completo está en el `README.md`).

In [ ]:
# @title Muestra tu caso de estudio
from IPython.display import display, Markdown

if not EQUIPO:
    print('⛔ Todavía no tienes tema asignado. Regresa a la sección 1.')
else:
    TEMA = EQUIPO['tema']
    INFO = CATALOGO[TEMA]
    display(Markdown('## %s · %s' % (TEMA, INFO['titulo'])))
    display(Markdown('**Incógnita:** $%s$  (%s)' % (INFO['incognita'], INFO['unidad'])))
    display(Markdown('**Ecuación del caso:**\n\n$$%s$$' % INFO['ecuacion']))
    display(Markdown('**Desafío principal:** %s' % INFO['desafio']))
    display(Markdown('**Métodos sugeridos:** %s' % ', '.join(INFO['metodos_sugeridos'])))
    _filas = ['| Parámetro | Valor nominal | Incertidumbre |', '|---|---|---|']
    for p, spec in INFO['params'].items():
        if spec[0] == 'u':
            _filas.append('| `%s` | %.4g | uniforme en [%.4g, %.4g] |' % (p, 0.5 * (spec[1] + spec[2]), spec[1], spec[2]))
        else:
            _filas.append('| `%s` | %.6g | ±%.1f%% |' % (p, spec[1], 100 * spec[2]))
    display(Markdown('\n'.join(_filas)))
    if INFO.get('polinomico'):
        display(Markdown('> 🔺 **Tema polinómico:** se exige **Müller o Bairstow** entre tus métodos.'))

---
# 4. Definición simbólica del problema

Con `sympy` el motor construye la función de tu caso y su **derivada** (o el **Jacobiano**,
si es un sistema), y las convierte en funciones rápidas con `lambdify`. Esta celda solo las
**muestra**, para que veas con qué expresión vas a trabajar.

> 📝 **Sobre los nombres.** En el **código** las variables se llaman con letras normales
> (`th1`, `omega`, `eps`), pero **en pantalla** se muestran como notación matemática
> ($\theta_1$, $\omega$, $\varepsilon$). Es el mismo símbolo: en el código usa siempre el
> nombre normal.


In [ ]:
# @title Definición simbólica y funciones numéricas
import numpy as np
import sympy as sp
from IPython.display import display, Markdown

MODELO = modelo(TEMA)          # viene del calificador (oculto)
ES_SISTEMA = MODELO['sistema']

if ES_SISTEMA:
    syms, exprs, sym_params = MODELO['sym']
    F_OFICIAL = exprs
    J_OFICIAL = sp.Matrix(exprs).jacobian(syms)
    display(Markdown('**Sistema de ecuaciones** $F(\\mathbf{x})=\\mathbf{0}$:'))
    for e in exprs:
        display(sp.Eq(e, 0))
    display(Markdown('**Jacobiano** $J=\\partial F/\\partial \\mathbf{x}$:'))
    display(J_OFICIAL)
else:
    x_sym, expr, sym_params = MODELO['sym']
    D_OFICIAL = sp.simplify(sp.diff(expr, x_sym))
    display(Markdown('**Ecuación** $f(%s)=0$:' % sp.latex(x_sym)))
    display(sp.Eq(expr, 0))
    display(Markdown('**Derivada** $f\'$:'))
    display(D_OFICIAL)

# Alias cómodos: ahora puedes escribir th1, omega, x, ... directamente
for _n, _s in zip(MODELO['vars'], (syms if ES_SISTEMA else [x_sym])):
    globals()[_n] = _s
print('Variables disponibles:', ', '.join(MODELO['vars']))
print('Parámetros del caso:', list(MODELO['params']))
print('Valores iniciales de referencia:', MODELO['info'].get('x0', MODELO['info'].get('intervalo')))

In [ ]:
# @title 🔍 Aísla la raíz (gráfica interactiva)
# Los métodos CERRADOS (bisección, falsa posición, punto fijo modificado) no
# pueden ni empezar si en los extremos del intervalo f(x) NO cambia de signo.
# Ese intervalo es el resultado de tu ANÁLISIS GRÁFICO, y es lo que se usa en
# toda la simulación.
import numpy as np
from IPython.display import display, Markdown

MIS_LIMITES = None       # <-- opcional: propón tu propio intervalo, p. ej. (0.005, 0.03)

if not EQUIPO:
    print("⛔ Todavía no tienes tema asignado.")
elif ES_SISTEMA:
    print("Tu caso es un SISTEMA de ecuaciones: se resuelve con Newton multivariable")
    print("(Jacobiano), no con métodos de intervalo, así que aquí no hay que aislar.")
else:
    _lo, _hi = INFO["intervalo"]
    _nom = {n: (s[1] if s[0] == "n" else 0.5 * (s[1] + s[2])) for n, s in INFO["params"].items()}
    _raices = oraculo_escalar_todas(lambda z: MODELO["f_arr"](z, _nom), _lo, _hi)
    _r = _raices[0] if _raices else float("nan")

    _xs = np.linspace(_lo, _hi, 800)
    _fig = visualizar.graficar_funcion(
        lambda z: MODELO["f_arr"](np.asarray(z, dtype=float), _nom),
        _xs, raiz=_r, nombre_var=MODELO["vars"][0],
        titulo="Tu caso: %s — %s" % (TEMA, INFO["titulo"]))
    _fig.show(config=visualizar.config_plotly())

    _fa = MODELO["f_arr"](np.array([_lo]), _nom)[0]
    _fb = MODELO["f_arr"](np.array([_hi]), _nom)[0]
    print("Intervalo del tema : (%.6g, %.6g)" % (_lo, _hi))
    print("f(extremo izquierdo) = %+.6g" % _fa)
    print("f(extremo derecho)   = %+.6g" % _fb)
    print("¿Cambia de signo?    : %s" % ("SÍ ✅ el intervalo aísla la raíz"
                                          if _fa * _fb < 0 else "NO ⚠️"))
    print("Raíz con valores nominales: %.8g" % _r)
    print("Número de cruces del eje dentro del intervalo: %d" % len(_raices))

    display(Markdown(
        "Verifica con tu equipo que la curva cruce el eje **una sola vez** dentro del intervalo.  \n"
        "Las perturbaciones del Monte Carlo son **pequeñas** (±1–5 %), así que este **mismo** "
        "intervalo contiene la raíz en **todas** las muestras de la simulación. **No hay que "
        "recalcularlo muestra por muestra.**"))

    if MIS_LIMITES is not None:
        _a, _b = MIS_LIMITES
        _za = MODELO["f_arr"](np.array([_a]), _nom)[0]
        _zb = MODELO["f_arr"](np.array([_b]), _nom)[0]
        _ok = _za * _zb < 0
        print("\nTu intervalo (%g, %g): f(a)=%+.4g  f(b)=%+.4g  →  %s"
              % (_a, _b, _za, _zb, "AÍSLA ✅" if _ok else "NO aísla ❌"))
        if _ok:
            print("   ¡Bien! Ancho = %.4g. Cuanto más angosto, menos iteraciones necesita bisección." % (_b - _a))


---
# 5. Elige los métodos que vas a comparar

Esta tarea es **teórica**: los métodos **ya vienen implementados** en el motor, así que no
programas nada. Tu trabajo es **elegir cuáles comparar** y **leer las gráficas**.

En las casillas de abajo **marca**:

1. **todos** los métodos **sugeridos** de tu caso (los viste en la sección 3), y
2. **al menos un contraejemplo**: un método que **no** esté sugerido, para contrastar en tu
   informe por qué en tu caso rinde distinto.

Las casillas están agrupadas **por familia** (cerrados, abiertos, híbrido, polinomios) para que
veas de inmediato qué estás comparando. El recuadro de la derecha te dice **en vivo** si tu
selección cumple la regla. En los casos de **sistema** (T14, T15, T19) no se pide contraejemplo:
sus dos métodos ya son los sugeridos.


In [ ]:
# @title 🎛️ Elige los métodos (marca las casillas) — esto es lo que se autocalifica
# Los métodos YA vienen implementados en el motor: esta tarea es teórica, así
# que no programas nada. Aquí solo eliges CUÁLES comparar.
#
# Marca las casillas:  TODOS los métodos sugeridos de tu caso  +  al menos UN
# método que NO esté sugerido (un contraejemplo, para contrastar en tu informe).
# En los casos de SISTEMA no se pide contraejemplo: sus dos métodos ya son los
# sugeridos.
import ipywidgets as widgets
from IPython.display import display

_DISPONIBLES = mapa_metodos(es_sistema(TEMA))    # {nombre: método} (lo pone el motor)
_SUGERIDOS = metodos_sugeridos_para(TEMA)

# Una casilla por método, agrupadas por familia para que se lean de un vistazo.
_FAMILIAS_UI = [("cerrado", "Cerrados (intervalo)"), ("abierto", "Abiertos"),
                ("hibrido", "Híbrido"), ("polinomios", "Polinomios"),
                ("desconocida", "Otros")]

CASILLAS = {}
_columnas = []
for _clave, _titulo in _FAMILIAS_UI:
    _miembros = [n for n in _DISPONIBLES if familia(n) == _clave]
    if not _miembros:
        continue
    _cajas = [widgets.HTML("<b>%s</b>" % _titulo)]
    for _n in _miembros:
        _cajas.append(widgets.Checkbox(value=(_n in _SUGERIDOS), description=_n,
                                       indent=False,
                                       layout=widgets.Layout(width="290px")))
        CASILLAS[_n] = _cajas[-1]
    _columnas.append(widgets.VBox(_cajas))

BT_SUGERIDOS = widgets.Button(description="Marcar solo los sugeridos")
OUT_METODOS = widgets.Output()
METODOS = {}

def _aplicar(*_):
    """METODOS = solo los métodos cuyas casillas están marcadas."""
    global METODOS
    METODOS = {n: _DISPONIBLES[n] for n in _DISPONIBLES if CASILLAS[n].value}
    with OUT_METODOS:
        OUT_METODOS.clear_output()
        if not METODOS:
            print("⚠️ No has marcado ningún método.")
            return
        for _n in METODOS:
            print("   %s %-28s familia: %s"
                  % ("⭐" if _n in _SUGERIDOS else "·", _n, familia(_n)))
        _faltan = [m for m in _SUGERIDOS if m not in METODOS]
        _extra = [n for n in METODOS if n not in _SUGERIDOS]
        print()
        if _faltan:
            print("⚠️ Te faltan métodos sugeridos: " + ", ".join(_faltan))
        else:
            print("✅ Incluyes todos los métodos sugeridos.")
        if _extra:
            print("· Contraejemplos (no sugeridos): " + ", ".join(_extra))
        elif not es_sistema(TEMA):
            print("⚠️ Agrega al menos un método NO sugerido (un contraejemplo).")

def _marcar_sugeridos(_):
    for _n, _cas in CASILLAS.items():
        _cas.value = (_n in _SUGERIDOS)

for _cas in CASILLAS.values():
    _cas.observe(_aplicar, names="value")
BT_SUGERIDOS.on_click(_marcar_sugeridos)

display(widgets.HBox(_columnas, layout=widgets.Layout(grid_gap="16px")))
display(BT_SUGERIDOS)
display(OUT_METODOS)
_aplicar()


---
# 6. Simulación de Monte Carlo

Se perturban los parámetros del caso (incertidumbre real) y se corre **cada método** sobre
las **mismas** $N$ muestras. Después se compara cada resultado contra la **raíz de
referencia** (el *oráculo*), muestra por muestra.

> La incertidumbre simula tolerancias de componentes, ruido de sensores, variación de
> fabricación y errores al *aislar* la raíz.


In [ ]:
# @title Ejecutar Monte Carlo
if not METODOS:
    raise ValueError("Primero elige al menos un método con el selector (sección 5).")

N_MC = 1000                              # tamaño de la simulación
SEMILLA_EQUIPO = semilla(TEMA, NC_JEFE)  # misma semilla para todos los métodos

print("Simulando %d muestras para el tema %s..." % (N_MC, TEMA))
RES = evaluar_tema(TEMA, METODOS, N=N_MC, seed=SEMILLA_EQUIPO)
print("Listo.\n")

print("%-30s %8s %8s %10s %10s %10s" % ("Método", "%éxito", "%diverg", "mediana", "media", "mediana_err"))
print("-" * 84)
for _m, _d in RES["metodos"].items():
    print("%-30s %7.1f%% %7.1f%% %10.0f %10.1f %10.1e" % (
        _m, 100 * _d["pct_exito"], 100 * _d["pct_diverge"],
        np.nanmedian(_d["n_iter"]), np.nanmean(_d["n_iter"]),
        np.nanmedian(_d["err_ref"])))
print("\nUn método \"CUENTA\" si acierta la raíz de referencia en >= %.0f%% de las muestras."
      % (100 * UMBRAL_EXITO))


---
# 7. Gráficos de desempeño

Primero el **histograma comparativo** y la **tasa de divergencia**; luego el
**box plot** (la vista más informativa) y la **distribución de la respuesta física**.

> 🖱️ Las gráficas son **interactivas**: acerca con la rueda, **selecciona puntos** con la caja
> o el lazo, y descarga la imagen con el icono de cámara.


In [ ]:
# @title 📊 Histograma de iteraciones y % de divergencia (interactivo)
_fig = visualizar.graficar_histograma(RES)
_fig.show(config=visualizar.config_plotly())


In [ ]:
# @title ⭐ Box plot comparativo de iteraciones (interactivo)
_fig = visualizar.graficar_boxplot(RES)
_fig.show(config=visualizar.config_plotly())

_met = list(RES["metodos"].keys())
_ord = sorted(_met, key=lambda m: np.nanmedian(RES["metodos"][m]["n_iter"]))
print("Ordenados por mediana de iteraciones (más rápido primero):")
for m in _ord:
    d = RES["metodos"][m]
    print("   %-30s mediana=%4.0f  dispersión(IQR)=%4.0f  fallos=%5.1f%%" % (
        m, np.nanmedian(d["n_iter"]),
        np.nanpercentile(d["n_iter"], 75) - np.nanpercentile(d["n_iter"], 25),
        100 * d["pct_diverge"]))


In [ ]:
# @title 📈 Distribución de la respuesta física ante la incertidumbre (interactivo)
_fig = visualizar.graficar_distribucion(RES, modelo=MODELO, es_sistema=ES_SISTEMA)
_fig.show(config=visualizar.config_plotly())

if not ES_SISTEMA:
    _rr = RES["raiz_ref"][np.isfinite(RES["raiz_ref"])]
    print("Respuesta física: media=%.6g  mediana=%.6g  desv=%.3g  min=%.6g  max=%.6g"
          % (np.mean(_rr), np.median(_rr), np.std(_rr), np.min(_rr), np.max(_rr)))
print("\nEsto responde: ¿qué tanto cambia la respuesta del sistema por la incertidumbre?")


---
# 8. Conclusiones de ingeniería

El notebook calcula dos respuestas a partir de **tus propios resultados** y te recuerda las
tres preguntas conceptuales. Todas se entregan en el **informe**; el profesor las revisa.

> 🎓 **La pregunta de la exposición:** *"Si estuvieran diseñando el firmware del controlador
> de este sistema mecatrónico en tiempo real, ¿qué método programarían y por qué?"*


In [ ]:
# @title 🎓 Preguntas de criterio (para el informe — NO se autocalifican)
from IPython.display import display, Markdown

_conteo = [m for m, d in RES["metodos"].items() if d["cuenta"]] or list(RES["metodos"].keys())
_rapido = min(_conteo, key=lambda m: np.nanmedian(RES["metodos"][m]["n_iter"]))
_fallador = max(RES["metodos"].keys(), key=lambda m: RES["metodos"][m]["pct_diverge"])

display(Markdown("### 1. Según TU simulación, ¿qué método necesitó MENOS iteraciones?"))
display(Markdown("**%s** (mediana %d iteraciones)." % (
    _rapido, np.nanmedian(RES["metodos"][_rapido]["n_iter"]))))

display(Markdown("### 2. ¿Qué método falló (no llegó a la raíz) en MÁS muestras?"))
display(Markdown("**%s** (%.1f%% de divergencia)." % (
    _fallador, 100 * RES["metodos"][_fallador]["pct_diverge"])))

display(Markdown("### 3. Preguntas conceptuales (para la exposición)"))
for _p in PREGUNTAS_CONCEPTO:
    display(Markdown("- **%s**: %s" % (_p["id"], _p["texto"])))

display(Markdown(
    "> 🎓 **La pregunta de la exposición:** *\"Si estuvieran diseñando el firmware del "
    "controlador de este sistema mecatrónico en tiempo real, ¿qué método programarían y por qué?\"*"))


---
# 9. Calificación automática y envío

Lo **único** que se califica solo es la **selección de métodos**: que hayas incluido los
**sugeridos** de tu caso y al menos un **contraejemplo**. El resto (informe, interpretación de
las gráficas y exposición) lo evalúa el **profesor a mano**.

Al enviar se reporta al servidor **qué métodos usaste** y si la selección fue correcta.


In [ ]:
# @title ✅ Calificar selección y enviar
NOTA_METODOS, DESGLOSE = calificar(TEMA, list(METODOS.keys()), RES)

print("=" * 74)
print("AUTOCALIFICACIÓN — SELECCIÓN DE MÉTODOS")
print("=" * 74)
print("Caso: %s — %s" % (TEMA, RES["titulo"]))
print("Seleccionaste  :", ", ".join(DESGLOSE["seleccion"]))
print("Sugeridos      :", ", ".join(DESGLOSE["sugeridos"]))
print("Faltantes      :", ", ".join(DESGLOSE["faltantes"]) if DESGLOSE["faltantes"] else "(ninguno) ✅")
print("Contraejemplos :", ", ".join(DESGLOSE["contraejemplos"]) if DESGLOSE["contraejemplos"]
      else ("(no se pide en casos de sistema)" if not DESGLOSE["requiere_contraejemplo"] else "(ninguno) ⚠️"))
print("Familias       :", ", ".join(DESGLOSE["familias"]))

if DESGLOSE["aprueba"]:
    print("\n✅ SELECCIÓN CORRECTA: incluiste los métodos correctos"
          + ("" if not DESGLOSE["requiere_contraejemplo"] else " y al menos un contraejemplo") + ".")
else:
    print("\n⚠️ SELECCIÓN INCOMPLETA:")
    if DESGLOSE["faltantes"]:
        print("   Te faltan los métodos sugeridos:", ", ".join(DESGLOSE["faltantes"]))
    if DESGLOSE["requiere_contraejemplo"] and not DESGLOSE["contraejemplos"]:
        print("   Agrega al menos un método NO sugerido (un contraejemplo).")

print("\n💡 Consejos:")
for _c in consejos(RES):
    print("   · %s" % _c)

# --- Envío ---
if not EQUIPO:
    print("\n⛔ No se puede enviar: no hay equipo registrado.")
else:
    try:
        _r = enviar_resultado(NC_JEFE, TEMA, list(METODOS.keys()), NOTA_METODOS,
                              correo=CORREO, alcance="equipo")
        print("\n✅ Enviado a la hoja de resultados:", _r.get("data", _r).get("mensaje", "ok"))
    except Exception as e:
        print("\n⚠️ No se pudo enviar todavía:", e)
        print("   Guarda el notebook y avisa al profesor. Tu resultado NO se perdió.")

print("\n--- Guarda el notebook en Drive (Archivo → Guardar) ---")
